# 04 Localization RGB-D Baseline

A compact PyTorch dataset/model skeleton for RGB-D UAV localization. Fill in the downloaded data paths before running.

In [ ]:
from pathlib import Path

RGB_DIR = Path("path/to/rgb")
DEPTH_DIR = Path("path/to/depth")
CSI_DIR = Path("path/to/csi/f60p0GHz_V")
LIMIT = 100

RGB_DIR, DEPTH_DIR, CSI_DIR

In [ ]:
import numpy as np

def load_position_label(csi_path):
    with np.load(csi_path, allow_pickle=False) as data:
        if "uav_pos" not in data.files:
            raise KeyError(f"{csi_path} is missing uav_pos")
        return data["uav_pos"].astype("float32").reshape(-1)[:3]

sample_files = sorted(CSI_DIR.glob("csi_*.npz"))[:LIMIT]
print(f"CSI samples: {len(sample_files)}")
if sample_files:
    print(load_position_label(sample_files[0]))

In [ ]:
# Optional RGB-D dataset/model skeleton. Install torch, torchvision, and pillow before running this cell.
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
    from torchvision import models, transforms
    from PIL import Image
except ImportError:
    torch = None
    print("Install torch, torchvision, and pillow to run the RGB-D localization baseline.")

if torch is not None:
    class LambdaRgbDepthLocalizationDataset(Dataset):
        def __init__(self, rgb_dir, depth_dir, csi_dir, limit=None):
            self.rgb_dir = Path(rgb_dir)
            self.depth_dir = Path(depth_dir)
            self.csi_files = sorted(Path(csi_dir).glob("csi_*.npz"))
            if limit is not None:
                self.csi_files = self.csi_files[:limit]
            self.rgb_transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
            self.depth_transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
            ])

        def __len__(self):
            return len(self.csi_files)

        def __getitem__(self, idx):
            csi_path = self.csi_files[idx]
            frame_id = csi_path.stem.replace("csi_", "")
            rgb_path = self.rgb_dir / f"img_{frame_id}.png"
            depth_path = self.depth_dir / f"depth_{frame_id}.npz"
            rgb = self.rgb_transform(Image.open(rgb_path).convert("RGB"))
            with np.load(depth_path, allow_pickle=False) as depth_npz:
                depth = depth_npz[depth_npz.files[0]].astype("float32")
            depth = self.depth_transform(Image.fromarray(depth, mode="F"))
            rgbd = torch.cat([rgb, depth], dim=0)
            target = torch.from_numpy(load_position_label(csi_path))
            return rgbd, target

    class RgbDepthRegressor(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = models.mobilenet_v2(weights=None)
            first = self.backbone.features[0][0]
            self.backbone.features[0][0] = nn.Conv2d(4, first.out_channels, first.kernel_size, first.stride, first.padding, bias=False)
            self.backbone.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, 3))

        def forward(self, rgbd):
            return self.backbone(rgbd)

    dataset = LambdaRgbDepthLocalizationDataset(RGB_DIR, DEPTH_DIR, CSI_DIR, limit=LIMIT)
    model = RgbDepthRegressor()
    print(f"dataset size={len(dataset)}")
    print(model)